In [ ]:
import numpy as np
import pylab as plt

In [ ]:
# set constants
OMEGA_PERTURBATION_SQUARED = 0.5 # technically this has units of angular frequency
OMEGA_KEPLER_SQUARED = (2. * np.pi) ** 2 # orbital angular frequency squared in (rad / yr)^2 for a circular orbit at r = 1 a.u.
TIDE = np.diag([1., 1., -2.])

In [ ]:
def a(r, t):
    return -OMEGA_KEPLER_SQUARED * r / np.linalg.norm(r) ** 3 \
           + 2. * OMEGA_PERTURBATION_SQUARED * (TIDE @ r)

In [ ]:
def leapfrog_start(r, v, h):
    """
    ## notes:
    - convention: move position is 0.5 * h into the future
    """
    return r + v * 0.5 * h, v + 0.

def leapfrog_step(r, v, t, a, h):
    v_next = v + a(r, t) * h
    return r + v_next * h, v_next

def leapfrog(r, v, a, h, n):
    """
    ## notes:
    - returns two sets of times, one for the positions and one for the velocities
    """
    d = len(r)
    assert len(v) == d
    rs, vs, ts = np.zeros((n, d)), np.zeros((n, d)), h * np.arange(n)
    rr, vv = leapfrog_start(r, v, h)
    rs[0], vs[0] = rr, vv
    for i in range(1, n):
        rr, vv = leapfrog_step(rr, vv, ts[i], a, h)
        rs[i], vs[i] = rr, vv
    return rs, ts + 0.5 * h, vs, ts

In [ ]:
h = 0.0001 # years
r0 = np.array([1.0, 0.0, 0.0]) # a.u.
foo = 0.0001 # radians
v0 = np.array([0.0, 2. * np.pi * np.cos(foo), 1.0]) # a.u. per year
rs, trs, vs, tvs = leapfrog(r0, v0, a, h, 2_000_000) # 200 years

In [ ]:
plt.plot(rs[:,0], rs[:, 2], "k-", alpha=0.5, lw=0.1)

In [ ]:
plt.plot(vs[:,0], vs[:, 2], "k-", alpha=0.5, lw=0.1)

In [ ]:
def get_positive_zero_crossings(xs, ts):
    ii = (xs <= 0.) & (np.roll(xs, 1) > 0.)
    ii[-1] = False # the last entry can't be a crossing
    idxs = np.arange(len(rs))[ii]
    crossings = 1. * np.zeros_like(idxs)
    for j, i in enumerate(idxs):
        crossings[j] = ts[i] - xs[i] * (ts[i+1] - ts[i]) / (xs[i+1] - xs[i])
    return crossings

In [ ]:
sos_times = get_positive_zero_crossings(rs[:, 2], trs) # z-axis!
print(sos_times)

In [ ]:
# use the correct times with the correct quantities
sos_xs = np.interp(sos_times, trs, rs[:, 0])
sos_ys = np.interp(sos_times, trs, rs[:, 1])
sos_vxs = np.interp(sos_times, tvs, vs[:, 0])
sos_vys = np.interp(sos_times, tvs, vs[:, 1])
sos_rhos = np.sqrt(sos_xs ** 2 + sos_ys ** 2)
sos_vrhos = (sos_vxs * sos_xs + sos_vys * sos_ys) / sos_rhos

In [ ]:
plt.scatter(sos_rhos, sos_vrhos, marker=".", c="k", s=1.0, alpha=0.75)

In [ ]:
def plot_one_orbit(r0, v0, a=a, h=h):
    rs, trs, vs, tvs = leapfrog(r0, v0, a, h, 2_000_000) # 200 years
    sos_times = get_positive_zero_crossings(rs[:, 2], trs) # z-axis!
    sos_xs = np.interp(sos_times, trs, rs[:, 0])
    sos_ys = np.interp(sos_times, trs, rs[:, 1])
    sos_vxs = np.interp(sos_times, tvs, vs[:, 0])
    sos_vys = np.interp(sos_times, tvs, vs[:, 1])
    sos_rhos = np.sqrt(sos_xs ** 2 + sos_ys ** 2)
    sos_vrhos = (sos_vxs * sos_xs + sos_vys * sos_ys) / sos_rhos
    return plt.scatter(sos_rhos, sos_vrhos, marker=".", c="k", s=1.0, alpha=0.75)

In [ ]:
for vr0 in np.arange(1.00, 1.49, 0.05):
    print(f"working on {vr0:0.2f}")
    v0 = np.array([vr0, 2. * np.pi * np.cos(foo), 0.6]) # a.u. per year
    plot_one_orbit(r0, v0)